# Meeting 2: generate and train

Generates the text-to-image pools and trains one run per pool, 5 seeds each, on the GPU. Every pool changes **one** setting of `m2-base`; everything else stays the same.

After each pool: the manifest, `pool.json` and contact sheet go to GitHub (the images stay local and can be regenerated identically), `model.pt` files go to Google Drive, and the run results go to GitHub.

Write your own prompts in `BASE_PROMPTS` before the first run, and don't change them afterwards: a pool refuses to resume under different settings. Afterwards, compare the runs on validation and set `POOL_RUN` in `m2_test.ipynb`.

In [4]:
import subprocess, sys
from getpass import getpass
from pathlib import Path
BRANCH = "excercises/2"  # the branch holding colab/ and runs/
REPO = Path("/content/dane_syntetyczne")
if not (REPO / ".git").exists():
    t = getpass("GitHub token (hidden): ")
    subprocess.run(["git", "clone", "--quiet", "-b", BRANCH, f"https://{t}@github.com/tmsz/dane_syntetyczne.git", str(REPO)], check=True)
if subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", BRANCH]).returncode:
    raise RuntimeError(f"Branch {BRANCH!r} is not on GitHub. On the Mac: git push -u origin {BRANCH}")
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "--quiet", "origin", BRANCH], check=True)
sys.path.insert(0, str(REPO / "colab"))
import colab_tools as ct
import importlib
importlib.reload(ct)  # re-running this cell picks up pulled changes
ct.authenticate()  # Drive + GitHub prompts now, before any download or training

✅ Google Drive mounted, GitHub push access OK


In [5]:
BASE_PROMPTS = [
    "Egyptian Mau cat, silver coat with small dark spots, sitting on a woven rug in daylight",
    "Egyptian Mau cat lying on a fabric sofa, spotted silver fur, green eyes, indoor lighting",
    "close-up of an Egyptian Mau cat face near a window, spotted forehead with M marking, natural light",
    "bronze Egyptian Mau cat walking across a wooden floor, small spots, casual home setting",
]

SIMPLE_PROMPT = [
    "Egyptian Mau cat",
]

NEGATIVE_PROMPT = (
    "drawing, painting, illustration, 3d render, cartoon, text, watermark, "
    "leopard, cheetah, jaguar, extra legs, deformed anatomy"
)

POOLS = {  # name: (prompts, extra synth-generate options, tags for synth-train)
    "m2-base": (BASE_PROMPTS, ["--negative-prompt", NEGATIVE_PROMPT], ["setting=base"]),
    "m2-guidance-2": (
        BASE_PROMPTS,
        ["--negative-prompt", NEGATIVE_PROMPT, "--guidance", "2"],
        ["guidance=2"],
    ),
    "m2-guidance-15": (
        BASE_PROMPTS,
        ["--negative-prompt", NEGATIVE_PROMPT, "--guidance", "15"],
        ["guidance=15"],
    ),
    "m2-steps-10": (
        BASE_PROMPTS,
        ["--negative-prompt", NEGATIVE_PROMPT, "--steps", "10"],
        ["steps=10"],
    ),
    "m2-prompt-simple": (
        SIMPLE_PROMPT,
        ["--negative-prompt", NEGATIVE_PROMPT],
        ["prompt=simple"],
    ),
    "m2-sdxl-turbo": (
        BASE_PROMPTS,
        ["--negative-prompt", NEGATIVE_PROMPT, "--model", "sdxl-turbo"],
        ["model=sdxl-turbo"],
    ),
}

In [6]:
ct.setup()  # pull, install, download data, check GPU
for name, (prompts, options, tags) in POOLS.items():
    ct.generate_pool(name, prompts, options)
    ct.save_to_github(f"Meeting 2: generate {name} in Colab")  # manifest, pool.json, contact sheet
    ct.train_run(name, "synthetic", pool=name, tags=tags)
    ct.check_devices([name])
    ct.save_models(name)  # model.pt -> Drive
    ct.save_to_github(f"Meeting 2: train {name} in Colab")  # everything else -> git

$ git pull --ff-only --quiet
$ /usr/bin/python3 -m pip install --quiet -e .
Data: 905 course photos in ./data
GPU: Tesla T4
GPU: Tesla T4

=== generate m2-base ===
$ synth-generate --name m2-base --device cuda --prompt Egyptian Mau cat, silver coat with small dark spots, sitting on a woven rug in daylight --prompt Egyptian Mau cat lying on a fabric sofa, spotted silver fur, green eyes, indoor lighting --prompt close-up of an Egyptian Mau cat face near a window, spotted forehead with M marking, natural light --prompt bronze Egyptian Mau cat walking across a wooden floor, small spots, casual home setting --negative-prompt drawing, painting, illustration, 3d render, cartoon, text, watermark, leopard, cheetah, jaguar, extra legs, deformed anatomy
m2-base: 98 of 110 images, t2i with stable-diffusion-v1-5/stable-diffusion-v1-5 on cuda (the first run downloads the model)
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please 